# SO-TAD Simple CNN Prototype on Colab

This notebook runs the lightweight `train_cnn.py` prototype added in this repo.

Use it to answer a narrow question first: can the SO-TAD dataset be loaded correctly and can a small model train on a single Colab GPU?

Important:
- Open this notebook from the repository version that already contains `train_cnn.py`.
- If your GitHub branch does not contain the modified files yet, upload this notebook together with the repo files to Colab or push your branch first.


In [ ]:
import os
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 1. Get the Code into Colab

Preferred path: clone your own branch/fork that contains the CNN prototype.

If you have not pushed your modified branch yet, skip this cell and upload the repo contents manually into `/content/so-tad-code`.


In [ ]:
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'
BRANCH = 'main'

# Replace REPO_URL/BRANCH above, then run.
# Example for the original upstream repo, only if your branch with train_cnn.py is there:
# REPO_URL = 'https://github.com/cccxy-299/so-tad.git'

!rm -rf /content/so-tad-code
!git clone --branch "$BRANCH" "$REPO_URL" /content/so-tad-code
%cd /content/so-tad-code
!ls


In [ ]:
%cd /content/so-tad-code
!python3 -m pip install -q torch torchvision opencv-python pillow tqdm
!python3 -m py_compile train_cnn.py dataset.py


## 2. Mount Google Drive and Point to the Dataset

Expected dataset layout:

```text
/content/drive/MyDrive/so-tad-dataset/
  Appendix.txt
  train/
    001.mp4
    002.mp4
    ...
  test/
    401.mp4
    402.mp4
    ...
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = '/content/drive/MyDrive/TCC/so-tad-dataset'
print('Dataset root:', DATASET_ROOT)
print('Exists:', os.path.isdir(DATASET_ROOT))


In [ ]:
!ls "$DATASET_ROOT"
!ls "$DATASET_ROOT/train" | head
!ls "$DATASET_ROOT/test" | head
!test -f "$DATASET_ROOT/Appendix.txt" && echo 'Appendix.txt found' || echo 'Appendix.txt missing'


## 3. Optional: Create a Small Working Subset

This avoids stressing Colab while you validate the pipeline. It copies only a small number of videos into a mini dataset.


In [ ]:
MINI_ROOT = '/content/so-tad-mini'

!mkdir -p "$MINI_ROOT/train" "$MINI_ROOT/test"
!cp "$DATASET_ROOT/Appendix.txt" "$MINI_ROOT/" || true
!ls "$DATASET_ROOT/train" | sort | head -n 100 | xargs -I{} cp "$DATASET_ROOT/train/{}" "$MINI_ROOT/train/"
!ls "$DATASET_ROOT/test" | sort | head -n 40 | xargs -I{} cp "$DATASET_ROOT/test/{}" "$MINI_ROOT/test/"
!echo 'Mini subset created at:' "$MINI_ROOT"
!echo 'Train videos:' $(ls "$MINI_ROOT/train" | wc -l)
!echo 'Test videos:' $(ls "$MINI_ROOT/test" | wc -l)


## 4. Train the Simple CNN Prototype

Start with the mini subset if you created it. If you want to train directly on the full extracted dataset, set `TRAIN_ROOT = DATASET_ROOT` instead.


In [ ]:
TRAIN_ROOT = '/content/so-tad-mini'
# TRAIN_ROOT = DATASET_ROOT

EPOCHS = 3
BATCH_SIZE = 8
IMAGE_SIZE = 96
FRAME_STRIDE = 45
MAX_FRAMES_PER_VIDEO = 8
TRAIN_VIDEO_LIMIT = 40
TEST_VIDEO_LIMIT = 20
NUM_WORKERS = 2

print('Training root:', TRAIN_ROOT)


In [ ]:
%cd /content/so-tad-code
!python3 train_cnn.py \
  --root "$TRAIN_ROOT" \
  --epochs "$EPOCHS" \
  --batch-size "$BATCH_SIZE" \
  --num-workers "$NUM_WORKERS" \
  --image-size "$IMAGE_SIZE" \
  --frame-stride "$FRAME_STRIDE" \
  --max-frames-per-video "$MAX_FRAMES_PER_VIDEO" \
  --train-video-limit "$TRAIN_VIDEO_LIMIT" \
  --test-video-limit "$TEST_VIDEO_LIMIT"


## 5. Inspect Outputs


In [ ]:
%cd /content/so-tad-code
!find runs -maxdepth 3 -type f | sort


## 6. If the First Run Works

Increase gradually:
- `TRAIN_VIDEO_LIMIT` and `TEST_VIDEO_LIMIT`
- `MAX_FRAMES_PER_VIDEO`
- `IMAGE_SIZE` to `128`
- `BATCH_SIZE` only if Colab VRAM allows it

Do not jump straight to the full article setup. The point here is to validate data loading, label assumptions, and whether the dataset has a learnable signal under a cheap baseline.
